# Transcription SIRA : affiner Whisper sur les voix d'Abidjan

On part de **Whisper `small`** (le modèle de secours du service vocal) et on l'adapte aux enregistrements
d'entraînement : Ange, Franck et Paul-Élie (≈ 41 min, 275 phrases chacun). Le but : mieux écrire les noms d'Abidjan
(Angré, CHU de Cocody, Zone 4, Adjamé 220 Logements…) et le parler ivoirien.

- **Encodeur gelé** : avec 40 min de voix, on n'adapte que le décodeur (ce qui écrit les mots). Le modèle garde ce
  qu'il sait déjà de l'écoute du français et n'apprend pas « par cœur » trois voix.
- **Validation** : 10 % des phrases ne sont jamais entendues à l'entraînement (choisies par `whisper_dataset.py`).
  Le meilleur passage est gardé d'après elles.
- **Sortie** : le modèle au format de ton service (CTranslate2 int8), prêt à copier dans
  `services/models/voice/whisper/sira-small-ft/`.
- **La vraie mesure se fait sur le PC** avec `npm run voice:bench` sur les lots de test (voix jamais entendues,
  phrases jamais vues). Attention : Franck est aussi dans l'entraînement, son score de test sera trop optimiste ;
  ta voix (lot du 19/10) et celle de Marie sont les vraies mesures.

**Avant de lancer :**
1. Sur le PC : `services/voice/.venv/Scripts/python services/voice/outils/whisper_dataset.py` → `dataset_whisper_sira.zip`.
2. Mets ce zip dans ton Google Drive, dossier **`SIRA_voix`** (le même que pour la voix Piper).
3. Colab : **Exécution → Modifier le type d'exécution → GPU T4**, puis **Exécuter tout** (≈ 20 à 30 min).

Sur **Kaggle** (≈ 30 h de GPU par semaine) : ajoute le zip comme *Dataset*, choisis l'accélérateur GPU T4, et mets
`DATA_ZIP` (cellule 1) sur le chemin du zip dans `/kaggle/input/…` ; le résultat sort dans `/kaggle/working`.

## 1. GPU, dossiers et réglages

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "Pas de GPU : change le type d'exécution")

if os.path.exists("/kaggle"):                      # Kaggle
    DATA_ZIP = "/kaggle/input/dataset-whisper-sira/dataset_whisper_sira.zip"   # ← à adapter au nom de ton Dataset
    OUT_DIR = "/kaggle/working"
else:                                              # Colab + Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ZIP = "/content/drive/MyDrive/SIRA_voix/dataset_whisper_sira.zip"
    OUT_DIR = "/content/drive/MyDrive/SIRA_voix/resultat"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ID = "openai/whisper-small"   # le modèle de secours du service vocal
WORK = "/tmp/sira_whisper"
EPOCHS = 5            # passages sur les ≈ 37 min d'entraînement ; le meilleur (validation) est gardé
LEARNING_RATE = 1e-5  # petit : on ajuste, on ne réapprend pas
BATCH = 16
print("Données :", DATA_ZIP, "→ résultat :", OUT_DIR)

## 2. Outils (versions figées : les mêmes transformers et CTranslate2 que sur le PC de SIRA)

In [ ]:
!pip -q install "transformers==5.17.0" "ctranslate2==4.8.2" "accelerate>=1.0" "jiwer>=3.0" soundfile

## 3. Données

In [ ]:
import csv, io, zipfile
import numpy as np, soundfile as sf

with zipfile.ZipFile(DATA_ZIP) as z:
    z.extractall(WORK)
print(open(f"{WORK}/rapport.txt", encoding="utf-8").read())
rows = list(csv.DictReader(open(f"{WORK}/metadata.csv", encoding="utf-8"), delimiter=";"))
train_rows = [r for r in rows if r["partie"] == "entrainement"]
valid_rows = [r for r in rows if r["partie"] == "validation"]
assert not {r["texte"] for r in train_rows} & {r["texte"] for r in valid_rows}, "une phrase de validation est dans l'entraînement"
print(len(train_rows), "phrases d'entraînement,", len(valid_rows), "de validation")

## 4. Modèle de départ : encodeur gelé, français, transcription

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.set_prefix_tokens(language="french", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.language = "french"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.model.encoder.requires_grad_(False)          # on n'adapte que le décodeur
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres entraînés : {trainable / 1e6:.0f} M sur {sum(p.numel() for p in model.parameters()) / 1e6:.0f} M")

## 5. Préparation : audio → spectrogramme, texte → jetons

In [ ]:
from dataclasses import dataclass

class Phrases(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]
        audio, sr = sf.read(f"{WORK}/{row['fichier']}", dtype="float32")
        assert sr == 16000, f"{row['fichier']} : {sr} Hz (whisper_dataset.py écrit du 16 kHz)"
        features = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
        return {"input_features": features, "labels": processor.tokenizer(row["texte"]).input_ids}

@dataclass
class Collator:
    def __call__(self, batch):
        features = processor.feature_extractor.pad([{"input_features": b["input_features"]} for b in batch], return_tensors="pt")
        labels = processor.tokenizer.pad([{"input_ids": b["labels"]} for b in batch], return_tensors="pt")
        ids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (ids[:, 0] == model.config.decoder_start_token_id).all():   # le modèle l'ajoute lui-même
            ids = ids[:, 1:]
        features["labels"] = ids
        return features

train_set, valid_set = Phrases(train_rows), Phrases(valid_rows)
print(train_set[0]["input_features"].shape, processor.tokenizer.decode(train_set[0]["labels"]))

## 6. Mesure : taux d'erreur de mots (même calcul que `benchmark_voice.py` : minuscules, sans ponctuation)

In [ ]:
import re, unicodedata, jiwer

def words(text):
    text = unicodedata.normalize("NFC", text.lower()).replace("’", "'")
    return " ".join(w for w in re.split(r"[\s\-']+", re.sub(r"[^\w'\- ]+", " ", text)) if w)

def wer_of(predictions, references):
    pairs = [(words(r), words(p)) for p, r in zip(predictions, references) if words(r)]
    return jiwer.wer([r for r, _ in pairs], [p for _, p in pairs])

def transcribe_all(rows, batch=16):
    model.eval()
    device = next(model.parameters()).device
    out = []
    for i in range(0, len(rows), batch):
        part = [Phrases(rows[i:i + batch])[k]["input_features"] for k in range(len(rows[i:i + batch]))]
        inputs = torch.tensor(np.array(part)).to(device, dtype=next(model.parameters()).dtype)
        with torch.no_grad():
            ids = model.generate(inputs, max_new_tokens=128)
        out += processor.batch_decode(ids, skip_special_tokens=True)
    return out

def compute_metrics(prediction):
    ids, labels = prediction.predictions, prediction.label_ids
    ids = np.where(ids != -100, ids, processor.tokenizer.pad_token_id)          # remplissage de l'évaluation
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    return {"wer": wer_of(processor.batch_decode(ids, skip_special_tokens=True),
                          processor.batch_decode(labels, skip_special_tokens=True))}

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
before = transcribe_all(valid_rows)
WER_BEFORE = wer_of(before, [r["texte"] for r in valid_rows])
print(f"Avant entraînement : WER validation = {WER_BEFORE:.1%}")

## 7. Entraînement (le meilleur passage d'après la validation est gardé)

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=f"{WORK}/checkpoints",
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    learning_rate=LEARNING_RATE, warmup_steps=30, num_train_epochs=EPOCHS,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    predict_with_generate=True, generation_max_length=128,
    fp16=torch.cuda.is_available(), logging_steps=10, report_to="none", remove_unused_columns=False,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_set, eval_dataset=valid_set,
                         data_collator=Collator(), compute_metrics=compute_metrics)
trainer.train()
for log in trainer.state.log_history:
    if "eval_wer" in log:
        print(f"passage {log['epoch']:.0f} : WER validation = {log['eval_wer']:.1%}")

## 8. Avant / après, phrase par phrase

In [ ]:
after = transcribe_all(valid_rows)
WER_AFTER = wer_of(after, [r["texte"] for r in valid_rows])
print(f"WER validation : {WER_BEFORE:.1%} avant → {WER_AFTER:.1%} après\n")
for row, b, a in list(zip(valid_rows, before, after))[:25]:
    if words(b) != words(a):
        print(f"[{row['locuteur']}] attendu : {row['texte']}\n   avant : {b.strip()}\n   après : {a.strip()}\n")

## 9. Export pour le service vocal (CTranslate2 int8, comme `whisper/small`)

Si le WER n'a pas baissé, ne remplace pas le modèle actuel : il faut plus de voix (d'autres personnes) plutôt que
plus de passages.

In [ ]:
import shutil
BEST = f"{WORK}/whisper-sira-small-ft"
trainer.save_model(BEST)
processor.save_pretrained(BEST)
# transformers 5 range la configuration audio dans processor_config.json ; le convertisseur attend
# preprocessor_config.json (et tokenizer.json) à côté du modèle : on les écrit aussi.
processor.feature_extractor.save_pretrained(BEST)
processor.tokenizer.save_pretrained(BEST)
CT2 = f"{WORK}/sira-small-ft"
shutil.rmtree(CT2, ignore_errors=True)
!ct2-transformers-converter --model {BEST} --output_dir {CT2} --quantization int8 --copy_files tokenizer.json preprocessor_config.json
with open(f"{CT2}/RESULTATS.txt", "w", encoding="utf-8") as f:
    f.write(f"Whisper small affiné SIRA\nWER validation : {WER_BEFORE:.1%} avant, {WER_AFTER:.1%} après\n"
            f"Entraînement : {len(train_rows)} phrases ; validation : {len(valid_rows)} phrases jamais entendues\n")
shutil.make_archive(f"{OUT_DIR}/sira-small-ft", "zip", CT2)
print("Prêt :", f"{OUT_DIR}/sira-small-ft.zip")

## 10. Sur le PC de SIRA

1. Dézippe `sira-small-ft.zip` dans `services/models/voice/whisper/sira-small-ft/` (on doit y voir `model.bin`).
2. Compare sur les voix de test, jamais entendues (toi et Marie ; Franck est dans l'entraînement) :
   ```
   npm run voice:bench -- --models small,sira-small-ft --audio services/voice/enregistrements/test/Marie_2026-10-24
   npm run voice:bench -- --models small,sira-small-ft --audio services/voice/enregistrements/2026-10-19
   ```
3. Si le WER et les lieux trouvés sont meilleurs, mets `VOICE_WHISPER_MODEL=sira-small-ft` (variable
   d'environnement du service vocal) : Whisper reste le secours de Parakeet, il entendra mieux les noms d'Abidjan.